# Lab | Langchain Evaluation

## Intro

Pick different sets of data and re-run this notebook. The point is for you to understand all steps involve and the many different ways one can and should evaluate LLM applications.

What did you learn? - Let's discuss that in class

## LangChain: Evaluation

### Outline:

* Example generation
* Manual evaluation (and debuging)
* LLM-assisted evaluation

In [1]:
from dotenv import load_dotenv, find_dotenv
import os
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')

### Example 1

#### Create our QandA application

In [2]:
# Install the right packages
%pip install -qU langchain langchain-classic langchain-openai langchain-huggingface

# UPDATED imports for LangChain v1+
from langchain_classic.chains import RetrievalQA
from langchain_openai import ChatOpenAI, OpenAI
from langchain_huggingface import HuggingFaceEmbeddings


In [3]:
!pip install -qU langchain-community
from langchain_community.document_loaders import CSVLoader
import os

# Load from the local data folder
file = 'data/OutdoorClothingCatalog_1000.csv'

try:
    loader = CSVLoader(file_path=file)
    data = loader.load()
    print(f"Loaded {len(data)} documents")
except Exception as e:
    print(f"Error loading CSV: {e}")

Loaded 1000 documents


In [4]:
# !pip install --upgrade --force-reinstall sentence-transformers

In [5]:
# Modern setup: stay on latest LangChain + integrations
%pip install -qU langchain langchain-community langchain-huggingface docarray langchain-text-splitters

# Imports (NO langchain.indexes here)
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import DocArrayInMemorySearch
from langchain_huggingface import HuggingFaceEmbeddings


In [6]:
# Imports (ensuring they are available)
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import DocArrayInMemorySearch
from langchain_huggingface import HuggingFaceEmbeddings

# 1. Use the documents already loaded in the 'data' variable
# loader = TextLoader("path/to/your/file.txt", encoding="utf-8")
# docs = loader.load()
docs = data  # 'data' holds the documents loaded from CSV earlier

# 2. Split into chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
)
splits = text_splitter.split_documents(docs)

# 3. Create embeddings (HF model example)
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# 4. Build the vector store directly instead of using VectorstoreIndexCreator
vectorstore = DocArrayInMemorySearch.from_documents(
    splits,
    embedding=embeddings,
)

# 5. If your old code used `index.vectorstore.as_retriever()`, now just:
retriever = vectorstore.as_retriever()

In [7]:
# Robust fix: uninstall and force reinstall to clear corrupted state
%pip uninstall -y langchain langchain-community langchain-core
%pip install -qU "langchain<1.0.0" langchain-community langchain-openai

import importlib
importlib.invalidate_caches()

from langchain.chains import RetrievalQA
from langchain_openai import ChatOpenAI
import os
import getpass
import langchain

# Monkey-patch the missing attribute to fix the library bug if it exists
if not hasattr(langchain, 'verbose'):
    langchain.verbose = False

# Ensure OpenAI API Key is set
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API Key: ")

llm = ChatOpenAI(temperature = 0.0)

# 'retriever' comes from the setup in the previous cell
qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    verbose=True,
    chain_type_kwargs = {
        "document_separator": "<<<<>>>>>"
    }
)

Found existing installation: langchain 1.1.0
Uninstalling langchain-1.1.0:
  Successfully uninstalled langchain-1.1.0
Found existing installation: langchain-community 0.4.1
Uninstalling langchain-community-0.4.1:
  Successfully uninstalled langchain-community-0.4.1
Found existing installation: langchain-core 1.1.0
Uninstalling langchain-core-1.1.0:
  Successfully uninstalled langchain-core-1.1.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-huggingface 1.1.0 requires langchain-core<2.0.0,>=1.1.0, but you have langchain-core 0.3.80 which is incompatible.
langgraph-prebuilt 1.0.5 requires langchain-core>=1.0.0, but you have langchain-core 0.3.80 which is incompatible.
langchain-classic 1.0.0 requires langchain-core<2.0.0,>=1.0.0, but you have langchain-core 0.3.80 which is incompatible.
langchain-classic 1.0.0 requires langchain-text-splitters<2.0.0,>

#### Coming up with test datapoints

In [8]:
data[10]

Document(metadata={'source': 'data/OutdoorClothingCatalog_1000.csv', 'row': 10}, page_content=": 10\nname: Cozy Comfort Pullover Set, Stripe\ndescription: Perfect for lounging, this striped knit set lives up to its name. We used ultrasoft fabric and an easy design that's as comfortable at bedtime as it is when we have to make a quick run out.\n\nSize & Fit\n- Pants are Favorite Fit: Sits lower on the waist.\n- Relaxed Fit: Our most generous fit sits farthest from the body.\n\nFabric & Care\n- In the softest blend of 63% polyester, 35% rayon and 2% spandex.\n\nAdditional Features\n- Relaxed fit top with raglan sleeves and rounded hem.\n- Pull-on pants have a wide elastic waistband and drawstring, side pockets and a modern slim leg.\n\nImported.")

In [9]:
data[11]

Document(metadata={'source': 'data/OutdoorClothingCatalog_1000.csv', 'row': 11}, page_content=': 11\nname: Ultra-Lofty 850 Stretch Down Hooded Jacket\ndescription: This technical stretch down jacket from our DownTek collection is sure to keep you warm and comfortable with its full-stretch construction providing exceptional range of motion. With a slightly fitted style that falls at the hip and best with a midweight layer, this jacket is suitable for light activity up to 20° and moderate activity up to -30°. The soft and durable 100% polyester shell offers complete windproof protection and is insulated with warm, lofty goose down. Other features include welded baffles for a no-stitch construction and excellent stretch, an adjustable hood, an interior media port and mesh stash pocket and a hem drawcord. Machine wash and dry. Imported.')

#### Hard-coded examples

In [10]:
!pip install -qU langchain langchain-core
from langchain_core.prompts import PromptTemplate

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-openai 0.3.35 requires langchain-core<1.0.0,>=0.3.78, but you have langchain-core 1.1.0 which is incompatible.
langchain-classic 1.0.0 requires langchain-text-splitters<2.0.0,>=1.0.0, but you have langchain-text-splitters 0.3.11 which is incompatible.


In [11]:
!pip install -qU langchain-core langchain-openai pydantic
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import BaseOutputParser
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field
import os
import getpass

# Ensure OpenAI API Key is set
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API Key: ")

examples = [
    {
        "query": "Do the Cozy Comfort Pullover Set\
        have side pockets?",
        "answer": "Yes"
    },
    {
        "query": "What collection is the Ultra-Lofty \
        850 Stretch Down Hooded Jacket from?",
        "answer": "The DownTek collection"
    }
]

# Define the prompt template
prompt_template = PromptTemplate(
    input_variables=["query"],
    template="Examples:\n"
             "1. Query: Do the Cozy Comfort Pullover Set have side pockets?\n"
             "   Answer: Yes\n"
             "2. Query: What collection is the Ultra-Lofty 850 Stretch Down Hooded Jacket from?\n"
             "   Answer: The DownTek collection\n"
             "Query: {query}\n"
             "Answer:"
)

# Define the output model
class Answer(BaseModel):
    answer: str = Field(description="The answer to the query")

# Create the output parser
class AnswerOutputParser(BaseOutputParser):
    def parse(self, text: str) -> Answer:
        # Split the response to get the answer
        if "Answer:" in text:
            answer = text.strip().split("Answer:")[-1].strip()
        else:
            answer = text.strip()
        return Answer(answer=answer)

# Initialize the LLM
llm = ChatOpenAI()

# Create the Chain using LCEL (LangChain Expression Language)
# This replaces LLMChain
chain = prompt_template | llm | AnswerOutputParser()

# Example query
query = "Is the Cozy Comfort Pullover Set available in different colors?"

# Run the chain
result = chain.invoke({"query": query})

# Print the result
print(result)

answer='Currently, the Cozy Comfort Pullover Set is only available in a neutral grey color.'


#### LLM-Generated examples

In [12]:
%pip install -qU \
  "langchain==0.0.179" \
  "langchain-community==0.0.27" \
  "langchain-openai==0.0.7" \
  "docarray==0.37.0" \
  "openai==0.27.7" \
  "tiktoken==0.4.0"


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: Cannot install langchain-openai==0.0.7 and openai==0.27.7 because these package versions have conflicting dependencies.
ERROR: ResolutionImpossible: for help visit https://pip.pypa.io/en/latest/topics/dependency-resolution/#dealing-with-dependency-conflicts


In [13]:
# Force uninstall the incorrect version
%pip uninstall -y langchain langchain-community langchain-core

# Install the official version (constrained to be less than 1.0.0 to avoid the bad package)
%pip install -U "langchain<1.0.0" langchain-community langchain-core langchain-openai

import importlib
importlib.invalidate_caches()

# Now import the chain
from langchain.evaluation.qa import QAGenerateChain

Found existing installation: langchain 1.1.0
Uninstalling langchain-1.1.0:
  Successfully uninstalled langchain-1.1.0
Found existing installation: langchain-community 0.3.31
Uninstalling langchain-community-0.3.31:
  Successfully uninstalled langchain-community-0.3.31
Found existing installation: langchain-core 1.1.0
Uninstalling langchain-core-1.1.0:
  Successfully uninstalled langchain-core-1.1.0
  Using cached langchain-0.3.27-py3-none-any.whl.metadata (7.8 kB)
  Using cached langchain_community-0.4.1-py3-none-any.whl.metadata (3.0 kB)
  Using cached langchain_core-1.1.0-py3-none-any.whl.metadata (3.6 kB)
  Using cached langchain_core-0.3.80-py3-none-any.whl.metadata (3.2 kB)
INFO: pip is looking at multiple versions of langchain-community to determine which version is compatible with other requirements. This could take a while.
  Using cached langchain_community-0.4-py3-none-any.whl.metadata (3.0 kB)
  Using cached langchain_community-0.3.31-py3-none-any.whl.metadata (3.0 kB)
INFO:

In [14]:
example_gen_chain = QAGenerateChain.from_llm(ChatOpenAI())

In [15]:
from langchain.chains import LLMChain
llm_chain = LLMChain(llm=llm, prompt=prompt_template)

/tmp/ipython-input-3993226932.py:2: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use `RunnableSequence, e.g., `prompt | llm`` instead.
  llm_chain = LLMChain(llm=llm, prompt=prompt_template)


In [16]:
new_examples = []
for t in data[:5]:
    # Extract the text content from the document
    txt = t.page_content

    # Run the chain to get the output
    # Note: .run() might return a string or a dict depending on the version/setup
    output = example_gen_chain.run(txt)

    # If the output is already a dictionary, use it directly.
    # Otherwise, parse the string output.
    if isinstance(output, dict):
        parsed_output = output
    else:
        parsed_output = example_gen_chain.output_parser.parse(output)

    # Append formatted result
    new_examples.append({"qa_pairs": parsed_output})

/tmp/ipython-input-1298795782.py:8: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use `invoke` instead.
  output = example_gen_chain.run(txt)


In [17]:
new_examples[0]

{'qa_pairs': {'query': "What is the approximate weight of the Women's Campside Oxfords per pair?",
  'answer': "The approximate weight of the Women's Campside Oxfords is 1 lb.1 oz. per pair."}}

In [18]:
data[0]

Document(metadata={'source': 'data/OutdoorClothingCatalog_1000.csv', 'row': 0}, page_content=": 0\nname: Women's Campside Oxfords\ndescription: This ultracomfortable lace-to-toe Oxford boasts a super-soft canvas, thick cushioning, and quality construction for a broken-in feel from the first time you put them on. \n\nSize & Fit: Order regular shoe size. For half sizes not offered, order up to next whole size. \n\nSpecs: Approx. weight: 1 lb.1 oz. per pair. \n\nConstruction: Soft canvas material for a broken-in feel and look. Comfortable EVA innersole with Cleansport NXT® antimicrobial odor control. Vintage hunt, fish and camping motif on innersole. Moderate arch contour of innersole. EVA foam midsole for cushioning and support. Chain-tread-inspired molded rubber outsole with modified chain-tread pattern. Imported. \n\nQuestions? Please contact us for any inquiries.")

In [19]:
d_flattened = [data['qa_pairs'] for data in new_examples]
d_flattened

[{'query': "What is the approximate weight of the Women's Campside Oxfords per pair?",
  'answer': "The approximate weight of the Women's Campside Oxfords is 1 lb.1 oz. per pair."},
 {'query': 'What are the dimensions of the small and medium sizes of the Recycled Waterhog Dog Mat, Chevron Weave?',
  'answer': 'The small size has dimensions of 18" x 28" and the medium size has dimensions of 22.5" x 34.5".'},
 {'query': "What features does the Infant and Toddler Girls' Coastal Chill Swimsuit, Two-Piece offer in terms of sun protection and comfort?",
  'answer': "The swimsuit offers UPF 50+ rated fabric for the highest sun protection, blocking 98% of the sun's harmful rays. It also features four-way-stretch and chlorine-resistant fabric, crossover no-slip straps, fully lined bottom, and bright colors, ruffles, and exclusive whimsical prints for comfort and style."},
 {'query': 'What is the fabric composition of the Refresh Swimwear, V-Neck Tankini Contrasts?',
  'answer': 'The body is mad

#### Combine examples

In [20]:
# examples += new_example
examples += d_flattened

In [21]:
examples[0]

{'query': 'Do the Cozy Comfort Pullover Set        have side pockets?',
 'answer': 'Yes'}

In [22]:
try:
    # qa was defined in the previous cell
    result = qa.invoke(examples[0]["query"])
    print(result)
except NameError:
    print("Error: 'qa' chain is not defined. Please execute the previous cell to initialize it.")



> Entering new RetrievalQA chain...

> Finished chain.
{'query': 'Do the Cozy Comfort Pullover Set        have side pockets?', 'result': 'The Cozy Cuddles Knit Pullover Set does not mention side pockets in the provided context.'}


### Manual Evaluation - Fun part

In [23]:
import langchain
langchain.debug = True

In [24]:
qa.invoke(examples[0]["query"])



> Entering new RetrievalQA chain...

> Finished chain.


{'query': 'Do the Cozy Comfort Pullover Set        have side pockets?',
 'result': 'The Cozy Cuddles Knit Pullover Set does not mention side pockets in the description provided.'}

In [25]:
# Turn off the debug mode
langchain.debug = False

### LLM assisted evaluation

In [26]:
examples += d_flattened

In [27]:
examples

[{'query': 'Do the Cozy Comfort Pullover Set        have side pockets?',
  'answer': 'Yes'},
 {'query': 'What collection is the Ultra-Lofty         850 Stretch Down Hooded Jacket from?',
  'answer': 'The DownTek collection'},
 {'query': "What is the approximate weight of the Women's Campside Oxfords per pair?",
  'answer': "The approximate weight of the Women's Campside Oxfords is 1 lb.1 oz. per pair."},
 {'query': 'What are the dimensions of the small and medium sizes of the Recycled Waterhog Dog Mat, Chevron Weave?',
  'answer': 'The small size has dimensions of 18" x 28" and the medium size has dimensions of 22.5" x 34.5".'},
 {'query': "What features does the Infant and Toddler Girls' Coastal Chill Swimsuit, Two-Piece offer in terms of sun protection and comfort?",
  'answer': "The swimsuit offers UPF 50+ rated fabric for the highest sun protection, blocking 98% of the sun's harmful rays. It also features four-way-stretch and chlorine-resistant fabric, crossover no-slip straps, ful

In [28]:
predictions = qa.batch(examples)



> Entering new RetrievalQA chain...


> Entering new RetrievalQA chain...


> Entering new RetrievalQA chain...


> Entering new RetrievalQA chain...


> Entering new RetrievalQA chain...


> Entering new RetrievalQA chain...

> Finished chain.


> Entering new RetrievalQA chain...

> Finished chain.


> Entering new RetrievalQA chain...

> Finished chain.


> Entering new RetrievalQA chain...

> Finished chain.


> Entering new RetrievalQA chain...

> Finished chain.


> Entering new RetrievalQA chain...

> Finished chain.


> Entering new RetrievalQA chain...

> Finished chain.

> Finished chain.

> Finished chain.

> Finished chain.

> Finished chain.

> Finished chain.


In [29]:
predictions

[{'query': 'Do the Cozy Comfort Pullover Set        have side pockets?',
  'answer': 'Yes',
  'result': 'The Cozy Cuddles Knit Pullover Set does not mention side pockets in the provided context.'},
 {'query': 'What collection is the Ultra-Lofty         850 Stretch Down Hooded Jacket from?',
  'answer': 'The DownTek collection',
  'result': 'The Ultra-Lofty 850 Stretch Down Hooded Jacket is from the DownTek collection.'},
 {'query': "What is the approximate weight of the Women's Campside Oxfords per pair?",
  'answer': "The approximate weight of the Women's Campside Oxfords is 1 lb.1 oz. per pair.",
  'result': "The approximate weight of the Women's Campside Oxfords per pair is 1 lb. 1 oz."},
 {'query': 'What are the dimensions of the small and medium sizes of the Recycled Waterhog Dog Mat, Chevron Weave?',
  'answer': 'The small size has dimensions of 18" x 28" and the medium size has dimensions of 22.5" x 34.5".',
  'result': 'The dimensions of the small size of the Recycled Waterhog 

In [30]:
from langchain.evaluation.qa import QAEvalChain

In [31]:
llm = ChatOpenAI(temperature=0)
eval_chain = QAEvalChain.from_llm(llm)

In [32]:
graded_outputs = eval_chain.evaluate(examples, predictions)

In [33]:
graded_outputs

[{'results': 'INCORRECT'},
 {'results': 'CORRECT'},
 {'results': 'CORRECT'},
 {'results': 'CORRECT'},
 {'results': 'CORRECT'},
 {'results': 'CORRECT'},
 {'results': 'CORRECT'},
 {'results': 'CORRECT'},
 {'results': 'CORRECT'},
 {'results': 'CORRECT'},
 {'results': 'CORRECT'},
 {'results': 'CORRECT'}]

In [34]:
for i, eg in enumerate(examples):
    print(f"Example {i}:")
    print("Question: " + predictions[i]['query'])
    print("Real Answer: " + predictions[i]['answer'])
    print("Predicted Answer: " + predictions[i]['result'])
    # print("Predicted Grade: " + graded_outputs[i]['text'])
    print()

Example 0:
Question: Do the Cozy Comfort Pullover Set        have side pockets?
Real Answer: Yes
Predicted Answer: The Cozy Cuddles Knit Pullover Set does not mention side pockets in the provided context.

Example 1:
Question: What collection is the Ultra-Lofty         850 Stretch Down Hooded Jacket from?
Real Answer: The DownTek collection
Predicted Answer: The Ultra-Lofty 850 Stretch Down Hooded Jacket is from the DownTek collection.

Example 2:
Question: What is the approximate weight of the Women's Campside Oxfords per pair?
Real Answer: The approximate weight of the Women's Campside Oxfords is 1 lb.1 oz. per pair.
Predicted Answer: The approximate weight of the Women's Campside Oxfords per pair is 1 lb. 1 oz.

Example 3:
Question: What are the dimensions of the small and medium sizes of the Recycled Waterhog Dog Mat, Chevron Weave?
Real Answer: The small size has dimensions of 18" x 28" and the medium size has dimensions of 22.5" x 34.5".
Predicted Answer: The dimensions of the sm

### Example 2
One can also easily evaluate your QA chains with the metrics offered in ragas

In [35]:
from langchain.indexes import VectorstoreIndexCreator
from langchain_community.document_loaders import TextLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import DocArrayInMemorySearch
from langchain.chains import RetrievalQA
from langchain_openai import ChatOpenAI
import os

# Download the text file if it doesn't exist
if not os.path.exists('nyc_text.txt'):
    !wget -q https://raw.githubusercontent.com/gkamradt/langchain-tutorials/main/data/nyc_text.txt -O nyc_text.txt

loader = TextLoader("nyc_text.txt")

# Use CPU instead of MPS (Mac)
embedding = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2", model_kwargs={'device': 'cpu'})

index = VectorstoreIndexCreator(
    embedding=embedding,
    vectorstore_cls=DocArrayInMemorySearch
).from_loaders([loader])

llm = ChatOpenAI(temperature=0)
qa_chain = RetrievalQA.from_chain_type(
    llm,
    retriever=index.vectorstore.as_retriever(),
    return_source_documents=True,
)

In [36]:
# testing it out

question = "How did New York City get its name?"
result = qa_chain.invoke({"query": question})
result["result"]

'New York City was named after the Duke of York (later King James II of England) in the 17th century. It was originally called New Amsterdam when it was a Dutch colony, but the English took control in 1664 and renamed it New York in honor of the Duke of York.'

In [37]:
result

{'query': 'How did New York City get its name?',
 'result': 'New York City was named after the Duke of York (later King James II of England) in the 17th century. It was originally called New Amsterdam when it was a Dutch colony, but the English took control in 1664 and renamed it New York in honor of the Duke of York.',
 'source_documents': []}

Now in order to evaluate the qa system we generated a few relevant questions. We've generated a few question for you but feel free to add any you want.

In [38]:
eval_questions = [
    "What is the population of New York City as of 2020?",
    "Which borough of New York City has the highest population?",
    "What is the economic significance of New York City?",
    "How did New York City get its name?",
    "What is the significance of the Statue of Liberty in New York City?",
]

eval_answers = [
    "8,804,190",
    "Brooklyn",
    "New York City's economic significance is vast, as it serves as the global financial capital, housing Wall Street and major financial institutions. Its diverse economy spans technology, media, healthcare, education, and more, making it resilient to economic fluctuations. NYC is a hub for international business, attracting global companies, and boasts a large, skilled labor force. Its real estate market, tourism, cultural industries, and educational institutions further fuel its economic prowess. The city's transportation network and global influence amplify its impact on the world stage, solidifying its status as a vital economic player and cultural epicenter.",
    "New York City got its name when it came under British control in 1664. King Charles II of England granted the lands to his brother, the Duke of York, who named the city New York in his own honor.",
    "The Statue of Liberty in New York City holds great significance as a symbol of the United States and its ideals of liberty and peace. It greeted millions of immigrants who arrived in the U.S. by ship in the late 19th and early 20th centuries, representing hope and freedom for those seeking a better life. It has since become an iconic landmark and a global symbol of cultural diversity and freedom.",
]

examples = [
    {"query": q, "ground_truths": [eval_answers[i]]}
    for i, q in enumerate(eval_questions)
]

In [39]:
examples

[{'query': 'What is the population of New York City as of 2020?',
  'ground_truths': ['8,804,190']},
 {'query': 'Which borough of New York City has the highest population?',
  'ground_truths': ['Brooklyn']},
 {'query': 'What is the economic significance of New York City?',
  'ground_truths': ["New York City's economic significance is vast, as it serves as the global financial capital, housing Wall Street and major financial institutions. Its diverse economy spans technology, media, healthcare, education, and more, making it resilient to economic fluctuations. NYC is a hub for international business, attracting global companies, and boasts a large, skilled labor force. Its real estate market, tourism, cultural industries, and educational institutions further fuel its economic prowess. The city's transportation network and global influence amplify its impact on the world stage, solidifying its status as a vital economic player and cultural epicenter."]},
 {'query': 'How did New York City

#### Introducing RagasEvaluatorChain

`RagasEvaluatorChain` creates a wrapper around the metrics ragas provides (documented [here](https://github.com/explodinggradients/ragas/blob/main/docs/metrics.md)), making it easier to run these evaluation with langchain and langsmith.

The evaluator chain has the following APIs

- `__call__()`: call the `RagasEvaluatorChain` directly on the result of a QA chain.
- `evaluate()`: evaluate on a list of examples (with the input queries) and predictions (outputs from the QA chain).
- `evaluate_run()`: method implemented that is called by langsmith evaluators to evaluate langsmith datasets.

lets see each of them in action to learn more.

In [40]:
result = qa_chain.invoke({"query": eval_questions[1]})
result["result"]

'As of the latest available data, Brooklyn is the borough of New York City with the highest population.'

In [41]:
key_mapping = {
    "query": "question",
    "result": "answer",
    "source_documents": "contexts"
}

result_updated = {}
for old_key, new_key in key_mapping.items():
    if old_key in result:
        result_updated[new_key] = result[old_key]


In [42]:
result_updated

{'question': 'Which borough of New York City has the highest population?',
 'answer': 'As of the latest available data, Brooklyn is the borough of New York City with the highest population.',
 'contexts': []}

In [43]:
# !pip install --no-cache-dir recordclass

In [44]:
!pip install ragas==0.1.9

In [45]:
from ragas.integrations.langchain import EvaluatorChain
# from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_relevancy,
    context_recall,
)

# create evaluation chains
faithfulness_chain   = EvaluatorChain(metric=faithfulness)
answer_rel_chain     = EvaluatorChain(metric=answer_relevancy)
context_rel_chain    = EvaluatorChain(metric=context_relevancy)
context_recall_chain = EvaluatorChain(metric=context_recall)

/usr/local/lib/python3.12/dist-packages/ragas/metrics/__init__.py:1: LangChainDeprecationWarning: As of langchain-core 0.3.0, LangChain uses pydantic v2 internally. The langchain_core.pydantic_v1 module was a compatibility shim for pydantic v1, and should no longer be used. Please update the code to import from Pydantic directly.

For example, replace imports like: `from langchain_core.pydantic_v1 import BaseModel`
with: `from pydantic import BaseModel`
or the v1 compatibility namespace if you are working in a code base that has not been fully upgraded to pydantic 2 yet. 	from pydantic.v1 import BaseModel

  from ragas.metrics._answer_correctness import AnswerCorrectness, answer_correctness
/usr/local/lib/python3.12/dist-packages/ragas/metrics/__init__.py:4: LangChainDeprecationWarning: As of langchain-core 0.3.0, LangChain uses pydantic v2 internally. The langchain.pydantic_v1 module was a compatibility shim for pydantic v1, and should no longer be used. Please update the code to impo

1. `__call__()`

Directly run the evaluation chain with the results from the QA chain. Do note that metrics like context_relevancy and faithfulness require the `source_documents` to be present.

In [46]:
# Recheck the result that we are going to validate.
result

{'query': 'Which borough of New York City has the highest population?',
 'result': 'As of the latest available data, Brooklyn is the borough of New York City with the highest population.',
 'source_documents': []}

**Faithfulness**

In [47]:
import nest_asyncio
nest_asyncio.apply()

eval_result = faithfulness_chain(result_updated)
eval_result["faithfulness"]

/tmp/ipython-input-2114669712.py:4: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use `invoke` instead.
  eval_result = faithfulness_chain(result_updated)


1.0

High faithfulness_score means that there are exact consistency between the source documents and the answer.

You can check lower faithfulness scores by changing the result (answer from LLM) or source_documents to something else.

In [48]:
fake_result = result_updated.copy()
fake_result["answer"] = "we are the champions"
eval_result = faithfulness_chain(fake_result)
eval_result["faithfulness"]

nan

**Context Relevancy**

In [49]:
# Prepare input with required keys: question, contexts, and ground_truth
input_data = result_updated.copy()
input_data["ground_truth"] = examples[1]["ground_truths"][0]

eval_result = context_recall_chain(input_data)
eval_result["context_recall"]

0.0

High context_recall_score means that the ground truth is present in the source documents.

You can check lower context recall scores by changing the source_documents to something else.

In [50]:
from langchain.schema import Document

# Prepare input with keys expected by Ragas: question, contexts, ground_truth
fake_input = {
    "question": result["query"],
    "contexts": ["I love christmas"],
    "ground_truth": examples[1]["ground_truths"][0]
}

eval_result = context_recall_chain(fake_input)
eval_result["context_recall"]

0.0

2. `evaluate()`

Evaluate a list of inputs/queries and the outputs/predictions from the QA chain.

In [51]:
# run the queries as a batch for efficiency
predictions = qa_chain.batch(examples)

# Prepare inputs for Ragas evaluation
# Map keys: query -> question, result -> answer, source_documents -> contexts
ragas_inputs = []
for i, pred in enumerate(predictions):
    ragas_inputs.append({
        "question": pred["query"],
        "answer": pred["result"],
        "contexts": [doc.page_content for doc in pred["source_documents"]],
        "ground_truth": examples[i]["ground_truths"][0]
    })

# evaluate using .batch()
print("evaluating...")
r = faithfulness_chain.batch(ragas_inputs)
r

evaluating...


[{'question': 'What is the population of New York City as of 2020?',
  'answer': "I don't know the exact population of New York City as of 2020.",
  'contexts': [],
  'ground_truth': '8,804,190',
  'faithfulness': 1.0},
 {'question': 'Which borough of New York City has the highest population?',
  'answer': 'As of the latest available data, Brooklyn is the borough of New York City with the highest population.',
  'contexts': [],
  'ground_truth': 'Brooklyn',
  'faithfulness': 1.0},
 {'question': 'What is the economic significance of New York City?',
  'answer': "New York City is a major financial hub, home to the New York Stock Exchange and many other financial institutions. It is also a center for various industries such as media, fashion, technology, and tourism. The city's diverse economy and large population make it a significant contributor to the overall economy of the United States.",
  'contexts': [],
  'ground_truth': "New York City's economic significance is vast, as it serves

In [52]:
# evaluate context recall
print("evaluating...")
# Use .batch() with the prepared inputs instead of .evaluate()
r = context_recall_chain.batch(ragas_inputs)
r

evaluating...


[{'question': 'What is the population of New York City as of 2020?',
  'answer': "I don't know the exact population of New York City as of 2020.",
  'contexts': [],
  'ground_truth': '8,804,190',
  'context_recall': 0.0},
 {'question': 'Which borough of New York City has the highest population?',
  'answer': 'As of the latest available data, Brooklyn is the borough of New York City with the highest population.',
  'contexts': [],
  'ground_truth': 'Brooklyn',
  'context_recall': 0.0},
 {'question': 'What is the economic significance of New York City?',
  'answer': "New York City is a major financial hub, home to the New York Stock Exchange and many other financial institutions. It is also a center for various industries such as media, fashion, technology, and tourism. The city's diverse economy and large population make it a significant contributor to the overall economy of the United States.",
  'contexts': [],
  'ground_truth': "New York City's economic significance is vast, as it se